# Heart Disease Prediction

Predicts whether a patient has heart disease using clinical features from a tabular dataset.
Compares Logistic Regression vs Decision Tree, with EDA, confusion matrix, ROC curve, and feature importance.

**Dataset:** `HeartDiseaseTrain-Test.csv` (upload when prompted).
The target column is `target` (1 = disease, 0 = no disease).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, roc_curve, auc

In [ ]:
# Upload the dataset when prompted
from google.colab import files
uploaded = files.upload()

In [ ]:
df = pd.read_csv('HeartDiseaseTrain-Test.csv')
print('Shape:', df.shape)
print('\nMissing values:')
print(df.isnull().sum())
df.head()

In [ ]:
# Drop duplicates and fill missing numeric values with column mean
df.drop_duplicates(inplace=True)
numeric_cols = df.select_dtypes(include='number').columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())
print('Clean shape:', df.shape)

## Exploratory Data Analysis

In [ ]:
sns.countplot(x='target', data=df)
plt.title('Heart Disease Class Distribution')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(df.select_dtypes(include='number').corr(), cmap='coolwarm', annot=False)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
sns.histplot(df['age'], kde=True)
plt.title('Age Distribution')
plt.tight_layout()
plt.show()

## Model Training

In [ ]:
X = df.drop('target', axis=1)
y = df['target']

# Convert any remaining categoricals and fill NaNs
X = pd.get_dummies(X)
X = X.fillna(X.mean(numeric_only=True))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f'Train: {len(X_train)} | Test: {len(X_test)}')

In [ ]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

print('Logistic Regression Accuracy:', accuracy_score(y_test, y_pred_lr))
print('Decision Tree Accuracy:      ', accuracy_score(y_test, y_pred_dt))

## Evaluation

In [ ]:
# Confusion Matrix — Logistic Regression
cm = confusion_matrix(y_test, y_pred_lr)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix — Logistic Regression')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curve
y_prob = lr.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f'Logistic Regression (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance — Logistic Regression (coefficients)
imp_lr = pd.DataFrame({'Feature': X.columns, 'Coefficient': lr.coef_[0]})
imp_lr = imp_lr.reindex(imp_lr['Coefficient'].abs().sort_values(ascending=False).index)
print('Top features (Logistic Regression):')
print(imp_lr.head(10).to_string(index=False))

In [ ]:
# Feature Importance — Decision Tree
imp_dt = pd.DataFrame({'Feature': X.columns, 'Importance': dt.feature_importances_})
imp_dt = imp_dt.sort_values('Importance', ascending=False)
print('Top features (Decision Tree):')
print(imp_dt.head(10).to_string(index=False))